# 实验二：Q-learning 与 SARSA 对比

## 实验目标

1. 基于 Gymnasium 创建 `CliffWalking-v1`。
2. 实现 epsilon-greedy。
3. 补全 Q-learning 与 SARSA 更新公式。
4. 使用曲线、箱线图、价值热力图和策略箭头图比较二者差异。

## 背景知识

###  CliffWalking（悬崖行走）环境

一个 **4 行 × 12 列 = 48 格**的网格世界。

```
 0   1   2   3   4   5   6   7   8   9  10  11
12  13  14  15  16  17  18  19  20  21  22  23
24  25  26  27  28  29  30  31  32  33  34  35
36  37  38  39  40  41  42  43  44  45  46  47
S   └────────── 悬崖 (37~46) ──────────┘   G
```

- **状态空间** `observation_space: Discrete(48)`：48 个格子，编号 0~47（行优先）。
- **动作空间** `action_space: Discrete(4)`：4 个方向 —— `0=↑ 上`、`1=→ 右`、`2=↓ 下`、`3=← 左`。
- **起点 S** = 状态 36（左下角），**终点 G** = 状态 47（右下角）。
- **奖励规则**：每走一步 **-1**（鼓励尽快到达）；掉入悬崖（37~46）得 **-100** 并被送回起点。
- **终止**：到达终点 G 时 `terminated=True`。



## 0. 环境


使用exp1的conda环境
```bash
conda activate rl-course
```


## 1. 导入依赖并创建环境


In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt

try:
    import gymnasium as gym
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("请先安装 gymnasium：%pip install -U gymnasium") from exc

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=3, suppress=True)

ENV_ID = "CliffWalking-v1"
SEED = 2026

def make_env(render_mode=None):
    return gym.make(ENV_ID, render_mode=render_mode)

env = make_env()
obs, info = env.reset(seed=SEED)
print("observation_space:", env.observation_space)
print("action_space:", env.action_space)
print("shape:", getattr(env.unwrapped, "shape", "unknown"))
env.close()


observation_space: Discrete(48)
action_space: Discrete(4)
shape: (4, 12)


## 2. 填空：核心算法


### 标准 TD 更新公式

Q-learning 使用 off-policy TD target：

$$
y_t = r_{t+1} + \gamma (1 - \mathbb{1}_{\text{terminal}}) \max_{a'} Q(s_{t+1}, a')
$$

$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[y_t - Q(s_t, a_t)\right]
$$

SARSA 使用 on-policy TD target：

$$
y_t = r_{t+1} + \gamma (1 - \mathbb{1}_{\text{terminal}}) Q(s_{t+1}, a_{t+1})
$$

$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[y_t - Q(s_t, a_t)\right]
$$


In [2]:
def epsilon_by_episode(episode, epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995):
    return max(epsilon_end, epsilon_start * (epsilon_decay ** episode))


def epsilon_greedy(q_table, state, epsilon, env, rng):
    if rng.random() < epsilon:
        return env.action_space.sample()
    return int(np.argmax(q_table[state]))


def q_learning_update(q_table, state, action, reward, next_state, terminated, alpha, gamma):
    best_next_q = np.max(q_table[next_state])
    target = reward + gamma * best_next_q * (1 - int(terminated))
    td_error = target - q_table[state, action]
    q_table[state, action] += alpha * td_error
    return td_error


def sarsa_update(q_table, state, action, reward, next_state, next_action, terminated, alpha, gamma):
    next_q = q_table[next_state, next_action]
    target = reward + gamma * next_q * (1 - int(terminated))
    td_error = target - q_table[state, action]
    q_table[state, action] += alpha * td_error
    return td_error


## 3. 训练循环


In [3]:
def train_q_learning(n_episodes=500, max_steps=300, alpha=0.5, gamma=0.99, seed=SEED):
    env = make_env()
    rng = random.Random(seed)
    q_table = np.zeros((env.observation_space.n, env.action_space.n), dtype=np.float32)
    history = {"return": [], "length": [], "epsilon": [], "mean_abs_td_error": []}

    for episode in range(n_episodes):
        state, info = env.reset(seed=seed + episode)
        state = int(state)
        epsilon = epsilon_by_episode(episode)
        episode_return, td_errors = 0.0, []

        for step in range(max_steps):
            action = epsilon_greedy(q_table, state, epsilon, env, rng)
            next_state, reward, terminated, truncated, info = env.step(action)
            next_state = int(next_state)
            td_error = q_learning_update(q_table, state, action, reward, next_state, terminated, alpha, gamma)
            td_errors.append(abs(float(td_error)))
            episode_return += float(reward)
            state = next_state
            if terminated or truncated:
                break

        history["return"].append(episode_return)
        history["length"].append(step + 1)
        history["epsilon"].append(epsilon)
        history["mean_abs_td_error"].append(float(np.mean(td_errors)) if td_errors else 0.0)

    env.close()
    return q_table, history


def train_sarsa(n_episodes=500, max_steps=300, alpha=0.5, gamma=0.99, seed=SEED):
    env = make_env()
    rng = random.Random(seed)
    q_table = np.zeros((env.observation_space.n, env.action_space.n), dtype=np.float32)
    history = {"return": [], "length": [], "epsilon": [], "mean_abs_td_error": []}

    for episode in range(n_episodes):
        state, info = env.reset(seed=seed + episode)
        state = int(state)
        epsilon = epsilon_by_episode(episode)
        action = epsilon_greedy(q_table, state, epsilon, env, rng)
        episode_return, td_errors = 0.0, []

        for step in range(max_steps):
            next_state, reward, terminated, truncated, info = env.step(action)
            next_state = int(next_state)
            next_action = epsilon_greedy(q_table, next_state, epsilon, env, rng)
            td_error = sarsa_update(q_table, state, action, reward, next_state, next_action, terminated, alpha, gamma)
            td_errors.append(abs(float(td_error)))
            episode_return += float(reward)
            state, action = next_state, next_action
            if terminated or truncated:
                break

        history["return"].append(episode_return)
        history["length"].append(step + 1)
        history["epsilon"].append(epsilon)
        history["mean_abs_td_error"].append(float(np.mean(td_errors)) if td_errors else 0.0)

    env.close()
    return q_table, history


## 4. 可视化工具


In [4]:
def moving_average(values, window=20):
    values = np.asarray(values, dtype=np.float32)
    if len(values) == 0:
        return values
    window = max(1, int(window))
    if len(values) < window:
        return np.full_like(values, values.mean())
    kernel = np.ones(window, dtype=np.float32) / window
    prefix = np.full(window - 1, values[:window].mean(), dtype=np.float32)
    return np.concatenate([prefix, np.convolve(values, kernel, mode="valid")])


def plot_returns(returns, lengths=None, window=20, title="Episode return"):
    returns = np.asarray(returns, dtype=np.float32)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(returns, color="#4C78A8", alpha=0.28, label="raw")
    ax.plot(moving_average(returns, window), color="#F58518", linewidth=2.2, label=f"{window}-episode mean")
    ax.set_title(title)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Return")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

    if lengths is not None:
        fig, ax = plt.subplots(figsize=(9, 3))
        ax.plot(lengths, color="#54A24B", linewidth=1.5)
        ax.set_title("Episode length")
        ax.set_xlabel("Episode")
        ax.set_ylabel("Steps")
        ax.grid(True, alpha=0.3)
        plt.show()


def plot_action_distribution(actions, title="Action distribution"):
    actions = np.asarray(actions, dtype=np.int64)
    if len(actions) == 0:
        print("没有动作数据可视化。")
        return
    labels, counts = np.unique(actions, return_counts=True)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar([str(x) for x in labels], counts, color="#54A24B")
    ax.set_title(title)
    ax.set_xlabel("Action")
    ax.set_ylabel("Count")
    ax.grid(True, axis="y", alpha=0.3)
    plt.show()


def plot_training_comparison(histories, window=25):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    metrics = [
        ("return", "Episode return"),
        ("length", "Episode length"),
        ("epsilon", "Epsilon"),
        ("mean_abs_td_error", "Mean |TD error|"),
    ]
    colors = {"Q-learning": "#4C78A8", "SARSA": "#F58518"}
    for ax, (key, title) in zip(axes.ravel(), metrics):
        for name, history in histories.items():
            values = np.asarray(history[key], dtype=np.float32)
            ax.plot(values, alpha=0.2, color=colors[name])
            ax.plot(moving_average(values, window), label=name, color=colors[name], linewidth=2)
        ax.set_title(title)
        ax.set_xlabel("Episode")
        ax.grid(True, alpha=0.3)
        ax.legend()
    fig.tight_layout()
    plt.show()


def evaluate_greedy(q_table, n_episodes=50, max_steps=300, seed=0):
    env = make_env()
    returns, lengths = [], []
    for episode in range(n_episodes):
        state, info = env.reset(seed=seed + episode)
        state = int(state)
        episode_return = 0.0
        for step in range(max_steps):
            action = int(np.argmax(q_table[state]))
            next_state, reward, terminated, truncated, info = env.step(action)
            state = int(next_state)
            episode_return += float(reward)
            if terminated or truncated:
                break
        returns.append(episode_return)
        lengths.append(step + 1)
    env.close()
    return np.asarray(returns), np.asarray(lengths)


def plot_policy_and_value(q_table, title="Policy and value"):
    env = make_env()
    shape = getattr(env.unwrapped, "shape", None)
    env.close()
    if shape is None:
        print("该环境无网格 shape。")
        return
    nrow, ncol = shape
    value = q_table.max(axis=1).reshape(nrow, ncol)
    policy = q_table.argmax(axis=1).reshape(nrow, ncol)
    symbols = {0: "↑", 1: "→", 2: "↓", 3: "←"}
    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(value, cmap="viridis")
    for r in range(nrow):
        for c in range(ncol):
            ax.text(c, r, symbols.get(int(policy[r, c]), "?"), color="white", ha="center", va="center", fontsize=14)
    ax.set_title(title)
    ax.set_xticks(range(ncol))
    ax.set_yticks(range(nrow))
    fig.colorbar(im, ax=ax, label="max_a Q(s,a)")
    plt.show()


## 5. 训练与分析


In [5]:
q_qlearning, hist_q = train_q_learning(n_episodes=500, seed=SEED)
q_sarsa, hist_s = train_sarsa(n_episodes=500, seed=SEED + 10000)

histories = {"Q-learning": hist_q, "SARSA": hist_s}
plot_training_comparison(histories, window=25)

for name, q in {"Q-learning": q_qlearning, "SARSA": q_sarsa}.items():
    returns, lengths = evaluate_greedy(q, n_episodes=50, seed=SEED + 20000)
    print(f"{name}: mean_return={returns.mean():.2f}, std={returns.std():.2f}, mean_length={lengths.mean():.1f}")
    plot_policy_and_value(q, title=f"{name}: greedy policy and value")


C:\Users\32967\AppData\Local\Temp\ipykernel_35368\844279981.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\32967\AppData\Local\Temp\ipykernel_35368\844279981.py:112: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Q-learning: mean_return=-13.00, std=0.00, mean_length=13.0
SARSA: mean_return=-17.00, std=0.00, mean_length=17.0


## 实验小结

- **Q-learning 的 TD target**：\(y_t = r_{t+1} + \gamma (1 - \mathbb{1}_{\text{terminal}}) \max_{a'} Q(s_{t+1}, a')\)。用下一状态的**最优动作价值**做 bootstrap，属于 **off-policy** 更新。
- **SARSA 的 TD target**：\(y_t = r_{t+1} + \gamma (1 - \mathbb{1}_{\text{terminal}}) Q(s_{t+1}, a_{t+1})\)。用**实际执行的下一动作** \(a_{t+1}\) 的 Q 值做 bootstrap，属于 **on-policy** 更新。
- **两者表现差异**（500 episode 训练 + 50 episode 贪心评估）：
  - **Q-learning**：`mean_return = -13.00`，`mean_length = 13.0` —— 学到贴悬崖边缘的最短路径。
  - **SARSA**：`mean_return = -17.00`，`mean_length = 17.0` —— 路径更长，但更远离悬崖、更保守。
  - **回报曲线**：二者均从约 \(-2500\) 逐步收敛；Q-learning 后期更贴近最优短路径回报。
  - **回合长度**：Q-learning 稳定在约 13 步；SARSA 更长且训练期波动更大。
  - **TD error**：填完公式后，|TD error| 在训练初期较大，随 Q 表收敛逐渐下降。
- **在 CliffWalking 中产生差异的原因**：悬崖格（37–46）惩罚 \(-100\)。Q-learning 按 \(\max_{a'} Q(s',a')\) 更新，会**乐观地**学到“贴边最短路径”；SARSA 按实际 \(\varepsilon\)-greedy 选到的 \(a_{t+1}\) 更新，探索时容易“看到”掉崖风险，因此学到**更远离悬崖的安全路径**。这正是 off-policy 最优策略 vs on-policy 实际策略的经典对比。
